In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json

def scrape_lords_diner_locations():
    url = "https://thelordsdiner.org/"
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, "html.parser")

    lines = [line.strip() for line in soup.get_text(separator="\n").split("\n") if line.strip()]
    county_map = {"wichita": "sedgwick", "pittsburg": "crawford"}

    pantries = []
    current_type = None
    entry = {}

    for i, line in enumerate(lines):

        if line.lower() in ["wichita broadway", "wichita hillside", "pittsburg kansas", "food trucks"]:
            current_type = line
            continue

        # Handle regular locations (1 line address + phone)
        if current_type and current_type.lower() != "food trucks":
            if re.match(r"\d{3,5} .*KS \d{5}", line):
                entry = {
                    "pantry_name": current_type,
                    "address": line,
                    "phone": "Not listed",
                    "raw_hours": None,
                    "requirements": "None specified",
                    "requirement_tags": [],
                    "requirement_explanations": "Serves hot meals every day",
                    "link": url,
                    "county": "unknown",
                    "city": ""
                }
                city = line.split(",")[0].split()[-1].lower()
                entry["city"] = city
                entry["county"] = county_map.get(city, "unknown")

                if i+1 < len(lines) and "Phone:" in lines[i+1]:
                    phone_match = re.search(r"\(\d{3}\)\s*\d{3}-\d{4}", lines[i+1])
                    if phone_match:
                        entry["phone"] = phone_match.group()
                pantries.append(entry)


        elif current_type and current_type.lower() == "food trucks":
            if "Center" in line:
                entry = {
                    "pantry_name": line,
                    "address": "",
                    "phone": "Not listed",
                    "raw_hours": None,
                    "requirements": "None specified",
                    "requirement_tags": ["no_id_required"],
                    "requirement_explanations": "Serves hot meals every day",
                    "link": url,
                    "county": "sedgwick",
                    "city": "wichita"
                }

                if i+1 < len(lines) and not lines[i+1].startswith("Meals served"):
                    entry["address"] = lines[i+1]

                if i+2 < len(lines) and "Meals served" in lines[i+2]:
                    entry["raw_hours"] = lines[i+2]
                pantries.append(entry)

    return pantries


data = scrape_lords_diner_locations()
save_path = "/content/drive/MyDrive/lords_diner_pantries.json"
with open(save_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved to your Google Drive: {save_path}")


Saved to your Google Drive: /content/drive/MyDrive/lords_diner_pantries.json


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json

def scrape_usd259_summer_food():
    url = "https://www.usd259.org/operations/nutrition-services/summerfood"
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, "html.parser")


    text = soup.get_text(separator="\n")
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]

    try:
        start_idx = next(i for i, ln in enumerate(lines) if "Free Meals - Dine-In Site Information" in ln)
    except StopIteration:
        return []

    entries = []
    pattern = re.compile(r"^(.*?)\s*-\s*(.*?)\s*,\s*(\d{5})$")
    i = start_idx + 1

    while i < len(lines):
        ln = lines[i]
        m = pattern.match(ln)
        if m:
            name = m.group(1).strip()
            address = f"{m.group(2).strip()}, KS {m.group(3)}"

            raw_hours = ""
            if i+2 < len(lines):
                raw_hours = f"{lines[i+1]} • {lines[i+2]}"
            entry = {
                "pantry_name": name,
                "address": address,
                "phone": "Not listed",
                "raw_hours": raw_hours,
                "requirements": "Children 1-18 eat free; adults may purchase",
                "requirement_tags": ["no_id_required"],
                "requirement_explanations": "Free meals for children ages 1–18; adults may purchase.",
                "link": url,
                "county": "sedgwick",
                "city": "wichita"
            }
            entries.append(entry)
            i += 4
        else:
            i +=1


    to_go_text = next((ln for ln in lines if "To-Go Site Information" in ln), None)
    if to_go_text:
        idx = lines.index(to_go_text)
        if idx+2 < len(lines):
            name = lines[idx+2].split("–")[0].strip()
            address = lines[idx+2].split("–")[1].strip()
            entry = {
                "pantry_name": name,
                "address": address,
                "phone": "Not listed",
                "raw_hours": lines[idx+3] if idx+3 < len(lines) else None,
                "requirements": "Pre-registration required; children 1-18 free",
                "requirement_tags": ["registration_required"],
                "requirement_explanations": "Parents must pre-register weekly",
                "link": url,
                "county": "sedgwick",
                "city": "wichita"
            }
            entries.append(entry)

    return entries

# Run scraper and save to Drive
data = scrape_usd259_summer_food()
save_path = "/content/drive/MyDrive/usd259_summer_food_sites.json"
with open(save_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved {len(data)} entries to Google Drive → {save_path}")


Saved 15 entries to Google Drive → /content/drive/MyDrive/usd259_summer_food_sites.json
